# 工程化分步复现手写数字识别

这个 notebook 是工程主入口，用来分步调用 `src/` 模块完成训练、评估和预测。它不作为提交给老师的独立材料；提交材料仍然保留 `submission_notebook.ipynb` 的自包含版本。

默认输出到 `outputs_runs/project_notebook`，不会覆盖上一版 99.8% 高分结果目录 `outputs_submission/`。

In [6]:
from pathlib import Path
import importlib
import sys

import pandas as pd
import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path(r"E:\ALL\学习\AI导论作业-识别手写数字")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.evaluate as evaluate_module
importlib.reload(evaluate_module)

from src.config import ExperimentConfig, ensure_project_paths
from src.data import create_dataloaders
from src.engine import fit
from src.evaluate import (
    collect_predictions,
    evaluate_external_holdouts,
    evaluate_mnist_c_zip,
    load_model_from_checkpoint,
    save_evaluation_bundle,
)
from src.model import build_model, count_model_parameters
from src.predict import PredictionImageDataset, predict_with_tta, write_predictions_csv
from src.train import set_seed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PROJECT_ROOT, DEVICE

(WindowsPath('e:/ALL/学习/AI导论作业-识别手写数字'), 'cuda')

## 运行开关

按需打开不同阶段。常见用法：

- 只训练：`RUN_TRAINING=True`
- 只评估旧模型：`LOAD_EXISTING_BEST_MODEL=True`, `RUN_EVALUATION=True`
- 输出多个测试集：`RUN_HOLDOUTS=True`
- 输出 MNIST-C corruption 汇总：`RUN_MNIST_C=True`
- 只预测考试图片：设置 `EXAM_IMAGE_DIR`，并打开 `RUN_PREDICTION=True`

注意：`max_samples=512` 只影响本 notebook 的快速 validation smoke test；正式的 holdout 评估会单独跑 MNIST/EMNIST/QMNIST 测试集。

In [7]:
RUN_TRAINING = False
RUN_EVALUATION = True
RUN_HOLDOUTS = True
RUN_MNIST_C = False
RUN_PREDICTION = False
LOAD_EXISTING_BEST_MODEL = True

HIGH_SCORE_CHECKPOINT = PROJECT_ROOT / "outputs_submission" / "checkpoints" / "best_model_state.pt"
EXAM_IMAGE_DIR = PROJECT_ROOT / "exam_data" / "test"

config = ExperimentConfig(
    project_root=PROJECT_ROOT,
    run_name="project_notebook",
    dataset_name="mnist",
    model_name="medium_cnn",
    batch_size=64,
    external_validation_batch_size=512,
    epochs=1,
    dropout=0.21672530847241062,
    optimizer_type="AdamW",
    scheduler_type="CosineAnnealingLR",
    learning_rate=0.0008398721379146775,
    weight_decay=6.602542933207749e-06,
    label_smoothing=0.03,
    max_samples=512,
    external_holdout_names=("mnist_test", "emnist_digits_test", "qmnist_test10k"),
    use_tta=True,
    tta_n=8,
)
paths = ensure_project_paths(config)
set_seed(config.seed)
config.to_dict()

{'project_root': 'e:\\ALL\\学习\\AI导论作业-识别手写数字',
 'dataset_name': 'mnist',
 'data_dir': 'e:\\ALL\\学习\\AI导论作业-识别手写数字\\data',
 'output_dir': 'e:\\ALL\\学习\\AI导论作业-识别手写数字\\outputs_runs\\project_notebook',
 'run_name': 'project_notebook',
 'model_name': 'medium_cnn',
 'batch_size': 64,
 'validation_split': 0.2,
 'validation_source': 'train_split',
 'image_size': 28,
 'num_classes': 10,
 'in_channels': 1,
 'learning_rate': 0.0008398721379146775,
 'epochs': 1,
 'seed': 42,
 'num_workers': 0,
 'pin_memory': False,
 'persistent_workers': False,
 'prefetch_factor': None,
 'dataloader_timeout': 0,
 'dropout': 0.21672530847241062,
 'label_smoothing': 0.03,
 'optimizer_type': 'AdamW',
 'scheduler_type': 'CosineAnnealingLR',
 'weight_decay': 6.602542933207749e-06,
 'use_amp': False,
 'allow_tf32': False,
 'use_early_stopping': False,
 'early_stopping_patience': 7,
 'early_stopping_min_delta': 0.0001,
 'rotation_degrees': 8.0,
 'translate_ratio': 0.08,
 'scale_min': 1.0,
 'scale_max': 1.0,
 'shear_degr

## 分步训练、评估、预测

下面的 cell 可以分开运行。默认使用小样本 `max_samples=512` 做 smoke test；完整训练时再关闭子采样并调整数据源。

In [8]:
train_loader, val_loader = create_dataloaders(config)
model = build_model(config).to(DEVICE)
total_params, trainable_params = count_model_parameters(model)

if RUN_TRAINING:
    history = fit(model, train_loader, val_loader, config=config, paths=paths, device=DEVICE)
    checkpoint_path = paths.checkpoints_dir / "best_model.pt"
elif LOAD_EXISTING_BEST_MODEL:
    checkpoint_path = HIGH_SCORE_CHECKPOINT
    history = {}
else:
    checkpoint_path = paths.checkpoints_dir / "best_model.pt"
    history = {}

{
    "checkpoint_path": str(checkpoint_path),
    "output_dir": str(paths.outputs_dir),
    "total_params": total_params,
    "trainable_params": trainable_params,
}

{'checkpoint_path': 'e:\\ALL\\学习\\AI导论作业-识别手写数字\\outputs_submission\\checkpoints\\best_model_state.pt',
 'output_dir': 'e:\\ALL\\学习\\AI导论作业-识别手写数字\\outputs_runs\\project_notebook',
 'total_params': 251562,
 'trainable_params': 251562}

## 结果输出

评估结果会写入 `outputs_runs/project_notebook/evaluation/`，预测结果会写入 `outputs_runs/project_notebook/predictions/`，不会影响 `outputs_submission/`。

In [9]:
if RUN_EVALUATION:
    eval_model, checkpoint_payload = load_model_from_checkpoint(checkpoint_path, config, DEVICE)
    images, y_true, y_pred = collect_predictions(eval_model, val_loader, device=DEVICE)
    validation_summary = save_evaluation_bundle(
        images=images,
        y_true=y_true,
        y_pred=y_pred,
        output_dir=paths.evaluation_dir / "validation_smoke",
        num_classes=config.num_classes,
    )
else:
    eval_model = None
    validation_summary = None

holdout_summaries = None
if RUN_HOLDOUTS:
    eval_model = eval_model or load_model_from_checkpoint(checkpoint_path, config, DEVICE)[0]
    holdout_summaries = evaluate_external_holdouts(
        eval_model,
        config=config,
        output_dir=paths.evaluation_dir / "holdouts",
        device=DEVICE,
    )

mnist_c_summary = None
if RUN_MNIST_C:
    eval_model = eval_model or load_model_from_checkpoint(checkpoint_path, config, DEVICE)[0]
    mnist_c_summary = evaluate_mnist_c_zip(
        eval_model,
        config=config,
        output_dir=paths.evaluation_dir / "mnist_c",
        device=DEVICE,
    )

metric_rows = []
if validation_summary is not None:
    metric_rows.append({
        "test_set": "validation_smoke",
        "accuracy": validation_summary["accuracy"],
        "macro_f1": None,
        "num_samples": validation_summary["num_samples"],
    })

for item in holdout_summaries or []:
    metric_rows.append({
        "test_set": item["name"],
        "accuracy": item["accuracy"],
        "macro_f1": item.get("macro_f1"),
        "num_samples": item["num_samples"],
    })

for item in mnist_c_summary or []:
    metric_rows.append({
        "test_set": f"mnist_c/{item['corruption']}",
        "accuracy": item["accuracy"],
        "macro_f1": item.get("macro_f1"),
        "num_samples": item["num_samples"],
    })

if mnist_c_summary:
    metric_rows.append({
        "test_set": "mnist_c/mean",
        "accuracy": sum(item["accuracy"] for item in mnist_c_summary) / len(mnist_c_summary),
        "macro_f1": sum(item["macro_f1"] for item in mnist_c_summary) / len(mnist_c_summary),
        "num_samples": sum(item["num_samples"] for item in mnist_c_summary),
    })

metrics_table = pd.DataFrame(metric_rows)
metrics_table["accuracy"] = metrics_table["accuracy"].map(lambda value: f"{value:.4f}")
metrics_table["macro_f1"] = metrics_table["macro_f1"].map(lambda value: "" if pd.isna(value) else f"{value:.4f}")
metrics_table

,test_set,accuracy,macro_f1,num_samples
0,validation_smoke,1.0000,,102
1,mnist_test,0.9973,0.9973,10000
2,emnist_digits_test,0.9973,0.9973,40000
3,qmnist_test10k,0.9973,0.9973,10000


In [10]:
if RUN_PREDICTION:
    if not EXAM_IMAGE_DIR.exists():
        raise FileNotFoundError(f"考试图片目录不存在: {EXAM_IMAGE_DIR}")
    pred_model, _ = load_model_from_checkpoint(checkpoint_path, config, DEVICE)
    prediction_dataset = PredictionImageDataset(
        EXAM_IMAGE_DIR,
        image_size=config.image_size,
        auto_invert=config.auto_invert,
    )
    prediction_loader = torch.utils.data.DataLoader(
        prediction_dataset,
        batch_size=config.batch_size,
        shuffle=False,
    )
    prediction_rows = []
    with torch.no_grad():
        for batch_images, filenames in prediction_loader:
            logits = predict_with_tta(pred_model, batch_images, config, DEVICE)
            predictions = logits.argmax(dim=1).cpu().tolist()
            prediction_rows.extend(zip(filenames, predictions))
    prediction_csv = paths.predictions_dir / "predictions.csv"
    write_predictions_csv(prediction_rows, prediction_csv)
else:
    prediction_csv = None

prediction_csv